In [1]:
#here we need to cleanup and focus on our aligned file with chimera to seperate and modify each file

In [ ]:
import os

def remove_t(input_path, output_path, ligand_resname="D", protein_chain="M"):
    os.makedirs(output_path, exist_ok=True)

    for filename in os.listdir(input_path):
        if filename.endswith(".pdb"):
            input_file = os.path.join(input_path, filename)
            output_file = os.path.join(output_path, filename)

            with open(input_file, "r") as f_in, open(output_file, "w") as f_out:
                for line in f_in:
                    if line.startswith("ATOM") and line[21].strip() == protein_chain:
                        f_out.write(line) 
                    elif line.startswith("HETATM") and line[17:20].strip() == ligand_resname:
                        f_out.write(line)

            print(f"Saved cleaned PDB: {output_file}")


In [6]:
import os

def filter_atom_lines(input_pdb, output_pdb):
    with open(input_pdb, 'r') as infile, open(output_pdb, 'w') as outfile:
        for line in infile:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                if len(line) < 22:
                    continue  # Skip malformed lines

                record_type = line[0:6].strip()
                chain_id = line[21].strip()
                residue_name = line[17:20].strip()

                if record_type == "ATOM" and chain_id != "T":
                    outfile.write(line)
                elif record_type == "HETATM" and residue_name == "D":
                    outfile.write(line)

def process_pdb_folder(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(input_dir):
        if filename.lower().endswith(".pdb"):
            input_path = os.path.join(input_dir, filename)

            # Replace 'aligned' with 'removed' in filename
            new_filename = filename.replace("aligned", "removed")
            output_path = os.path.join(output_dir, new_filename)

            try:
                filter_atom_lines(input_path, output_path)
                print(f"Processed: {filename} -> {new_filename}")
            except Exception as e:
                print(f"Failed: {filename} — {e}")

# Run for all aligned files
input_directory = "/home/k_ensafitakaldani001_umb_edu/BLAST/clean_code/1"
output_directory = "/home/k_ensafitakaldani001_umb_edu/BLAST/clean_code/2"

process_pdb_folder(input_directory, output_directory)


Processed: aligned_104_3GR6_renamed_vs_1NHG_renamed.pdb -> removed_104_3GR6_renamed_vs_1NHG_renamed.pdb


In [ ]:
#chain split with chimera when 

In [ ]:
import os
import math
import csv

def parse_pdb_atoms_chainT_with_residueD(pdb_file, ligand_resname="D", ligand_chain="T", protein_chain="M"):
    ligand_atoms = []
    protein_c_alphas = []
    
    with open(pdb_file, "r") as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                atom_name = line[12:16].strip()
                res_name = line[17:20].strip()
                chain_id = line[21].strip()
                res_seq = int(line[22:26].s trip())
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                
                atom_data = {
                    "atom_name": atom_name,
                    "res_name": res_name,
                    "chain_id": chain_id,
                    "res_seq": res_seq,
                    "coord": (x, y, z)
                }
                
                if chain_id == protein_chain and atom_name == "CA":
                    protein_c_alphas.append(atom_data)
                elif chain_id == ligand_chain and res_name == ligand_resname:
                    ligand_atoms.append(atom_data)
    
    return ligand_atoms, protein_c_alphas

def compute_all_distances(ligand_atoms, protein_c_alphas, cutoff=5.0):
    close_contacts = []
    for lig_atom in ligand_atoms:
        for ca_atom in protein_c_alphas:
            lx, ly, lz = lig_atom["coord"]
            px, py, pz = ca_atom["coord"]

            
            ## Euclidean distance formula 
            dist = math.sqrt((lx - px)**2 + (ly - py)**2 + (lz - pz)**2)

            
            if dist < cutoff:
                close_contacts.append({
                    "ligand_residue": lig_atom["res_name"],
                    "ligand_resseq": lig_atom["res_seq"],
                    "ligand_atom": lig_atom["atom_name"],
                    "ligand_coord": lig_atom["coord"],
                    "protein_residue": ca_atom["res_name"],
                    "protein_resseq": ca_atom["res_seq"],
                    "protein_coord": ca_atom["coord"],
                    "distance": round(dist, 3)
                })
    return close_contacts


In [ ]:

def save_contacts_to_csv(contacts, output_csv):
    with open(output_csv, "w", newline="") as csvfile:
        fieldnames = ["ligand_residue", "ligand_resseq", "ligand_atom", "ligand_coord",
                      "protein_residue", "protein_resseq", "protein_coord", "distance"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for contact in contacts:
            writer.writerow(contact)

def process_pdb_folder(input_dir, output_dir, ligand_resname="D", ligand_chain="T", protein_chain="M", cutoff=5.0):
    os.makedirs(output_dir, exist_ok=True)
    
    for filename in os.listdir(input_dir):
        if filename.endswith(".pdb"):
            pdb_path = os.path.join(input_dir, filename)
            output_csv_name = os.path.splitext(filename)[0] + ".csv"
            output_csv_path = os.path.join(output_dir, output_csv_name)
            
            try:
                ligand_atoms, protein_c_alphas = parse_pdb_atoms_chainT_with_residueD(
                    pdb_path, ligand_resname, ligand_chain, protein_chain
                )
                results = compute_all_distances(ligand_atoms, protein_c_alphas, cutoff)
                save_contacts_to_csv(results, output_csv_path)
                print(f"Processed: {filename} -> {output_csv_name} ({len(results)} contacts)")
            except Exception as e:
                print(f"Failed to process {filename}: {e}")

# === Example Run ===
input_folder = "/content/drive/MyDrive/chimeraalign/aligned_pdbs"
output_folder = "/content/close_contact_outputs"
#if output_folder doesnt exist create it
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    
process_pdb_folder(input_folder, output_folder)


In [ ]:
pubchem_cid_y_unique = {
    5564: 'TCL', 4993: 'CP6', 11442891: '627', 9914412: '447', 5329102: 'B49',
    46398810: 'DQX', 25195294: 'LDN', 392622: 'RIT', 5328940: 'DB8', 11626560: 'VGH',
    10138980: 'GR9', 16007391: 'YJA', 462382: 'LDZ', 16722832: '1M3', 9829523: '2K2',
    11364421: 'R78', 16722836: '2TA', 44556162: 'SVE', 46843772: '4WG', 439530: 'PUY',
    119081415: 'V0G', 71604307: 'VYJ', 54576299: 'A9I', 46866319: 'S59', 72716071: 'QS0',
    2179: 'ASW', 51038269: 'T3X', 24826799: '0LI', 2247: 'XB7', 25183872: '6V8',
    10247560: 'XCF'
}
